# 🎬 पावर ओरिएंटर हब – फ़ोटो से हिलता वीडियो (फ़्री)
**ये मशीन आपके अपने अकाउंट के फ़्री GPU पर चलती है।** आपकी फ़ोटो किसी और के पास नहीं जाती।

### चालू करने का तरीक़ा
- **Colab में:** ऊपर `Runtime → Change runtime type → T4 GPU` चुनो, फिर नीचे वाले सेल का ▶ दबाओ।
- **Kaggle में:** दाईं ओर `Settings → Accelerator → GPU T4 x2` और `Internet → On` करो, फिर ▶ दबाओ।
- 8–15 मिनट बाद नीचे एक **`https://….gradio.live`** लिंक आएगा। उसे खोलो, वहीं वीडियो बनाओ।

### ⏱️ सच्ची जानकारी
- पहली बार मॉडल डाउनलोड होने में 5–10 मिनट लगते हैं।
- एक 5 सेकंड का वीडियो: लगभग **4–6 मिनट (480p)** या **9–12 मिनट (720p)**।
- फ़्री GPU की हफ़्ते की सीमा होती है (Kaggle में 30 घंटे)। काम पूरा हो जाए तो सेशन बंद कर दो।

### 🚫 नियम
असली लोगों के नक़ली वीडियो बनाना, किसी को बदनाम करना, अश्लील या हिंसक कंटेंट बनाना **मना है**। हर वीडियो पर "AI Generated" लिखा आता है।

*Claude AI की मदद से बनाया गया। इस्तेमाल किए गए मॉडल: Wan 2.2 (Apache 2.0)।*

In [ ]:
# ===== पावर ओरिएंटर हब – एक बटन वाली मशीन =====
import os, sys, subprocess, shutil, glob, json, time, uuid
def sh(c): return subprocess.run(c, shell=True)
# ---- 1) कौन सी जगह: Colab या Kaggle ----
if os.path.exists('/kaggle'): BASE, PLACE = '/kaggle/temp', 'Kaggle'
elif os.path.exists('/content'): BASE, PLACE = '/content', 'Colab'
else: BASE, PLACE = os.path.expanduser('~/poh'), 'Computer'
os.makedirs(BASE, exist_ok=True)
print('📍 जगह:', PLACE)
# ---- 2) GPU जाँच ----
g = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip()
if not g:
    raise SystemExit('❌ GPU नहीं मिला!\n Colab: Runtime → Change runtime type → T4 GPU\n Kaggle: Settings → Accelerator → GPU T4 x2\n फिर दोबारा ▶ दबाओ।')
print('✅ GPU:', g.replace('\n', ' | '))
# ---- 3) मशीन इंस्टॉल (पहली बार 5-8 मिनट) ----
WG = BASE + '/Wan2GP'
if not os.path.exists(WG + '/wgp.py'):
    print('⏳ मशीन इंस्टॉल हो रही है...')
    sh(f'git clone -q https://github.com/deepbeepmeep/Wan2GP {WG}')
os.chdir(WG)
if not os.path.exists('.poh_ok'):
    sh("grep -viE '^(torch|torchvision|torchaudio)([=<> ]|$)' requirements.txt > req_poh.txt")
    sh('pip install -q -r req_poh.txt')
    sh('pip install -q -U "protobuf>=5.28"')
    sh("pip install -q $(grep -iE '^(gradio|huggingface)' requirements.txt)")
    open('.poh_ok', 'w').write('ok')
print('✅ मशीन तैयार')
# ---- 4) "AI Generated" निशान ----
from PIL import Image, ImageDraw, ImageFont
def label_png(path, w):
    h = max(28, w // 40); img = Image.new('RGBA', (w, h + 16), (0, 0, 0, 0)); d = ImageDraw.Draw(img)
    try: f = ImageFont.truetype(glob.glob('/usr/share/fonts/**/DejaVuSans*.ttf', recursive=True)[0], h - 8)
    except Exception: f = ImageFont.load_default()
    t = 'AI Generated | Power Orientor Hub'; tw = d.textlength(t, font=f)
    d.rounded_rectangle((w - tw - 34, 6, w - 10, h + 8), 8, fill=(0, 0, 0, 120))
    d.text((w - tw - 22, 9), t, font=f, fill=(255, 255, 255, 215)); img.save(path)
OUT = BASE + '/poh_videos'; os.makedirs(OUT, exist_ok=True)
NEG = 'text, watermark, subtitles, extra fingers, deformed face, blurry, bright pink colors, oversaturated'
LORA = 'https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/FastWan/Wan2_2_5B_FastWanFullAttn_lora_rank_128_bf16.safetensors'
def banao(photo, prompt, quality, agree, progress=None):
    import gradio as gr
    if not agree: raise gr.Error('पहले नियम वाले बॉक्स पर ✓ लगाओ।')
    if photo is None: raise gr.Error('पहले फ़ोटो डालो।')
    prompt = (prompt or '').strip() or 'subtle natural cinematic movement, gentle camera push in'
    jid = uuid.uuid4().hex[:8]; jd = f'{OUT}/job_{jid}'; os.makedirs(jd)
    img = Image.open(photo).convert('RGB'); img.save(f'{jd}/in.png')
    land = img.width >= img.height
    res = {'480': ('832x480' if land else '480x832'), '720': ('1280x720' if land else '720x1280')}[quality]
    st = {"model_type": "ti2v_2_2", "image_mode": 0, "prompt": prompt + ', cinematic, natural colors, smooth motion',
          "negative_prompt": NEG, "resolution": res, "video_length": 121, "seed": int(time.time()) % 100000,
          "num_inference_steps": 6, "guidance_scale": 1, "guidance_phases": 1, "flow_shift": 5, "sample_solver": "unipc",
          "repeat_generation": 1, "multi_prompts_gen_type": "PG", "activated_loras": [LORA], "loras_multipliers": "",
          "image_prompt_type": "S", "video_prompt_type": "", "audio_prompt_type": "", "sliding_window_size": 121,
          "sliding_window_overlap": 1, "video_quality": "libx264_8", "image_start": f'{jd}/in.png'}
    json.dump(st, open(f'{jd}/set.json', 'w'))
    t = time.time()
    sh(f'cd {WG} && CUDA_VISIBLE_DEVICES=0 python wgp.py --process {jd}/set.json --output-dir {jd}/raw --attention sdpa --profile 4 > {jd}/log.txt 2>&1')
    v = sorted(glob.glob(f'{jd}/raw/**/*.mp4', recursive=True), key=os.path.getmtime)
    if not v: raise gr.Error('वीडियो नहीं बना। दोबारा कोशिश करो। (log: ' + open(f'{jd}/log.txt').read()[-200:] + ')')
    W_, H_ = (1920, 1080) if land else (1080, 1920)
    label_png(f'{jd}/lab.png', W_)
    final = f'{OUT}/POH_{jid}.mp4'
    sh(f'ffmpeg -v error -y -i "{v[-1]}" -i {jd}/lab.png -filter_complex "[0:v]scale={W_}:{H_}:flags=lanczos,unsharp=5:5:0.7:3:3:0.3,fps=24[b];[b][1:v]overlay=0:H-h-10" '
       f'-c:v libx264 -preset veryfast -crf 18 -pix_fmt yuv420p -an {final}')
    return final, f'✅ तैयार! {(time.time()-t)/60:.1f} मिनट लगे। नीचे वीडियो पर ⋮ या ⬇ दबाकर डाउनलोड करो।'
# ---- 5) ब्रांड वाली स्क्रीन ----
import gradio as gr
CSS = '.gradio-container{max-width:760px!important;margin:auto} #hd{text-align:center} #go{background:#e8a317!important;color:#111!important;font-weight:700}'
with gr.Blocks(title='Power Orientor Hub', css=CSS, theme=gr.themes.Soft()) as app:
    gr.Markdown('# 🎬 पावर ओरिएंटर हब\n### फ़ोटो से हिलता वीडियो – फ़्री, आपके अपने GPU पर', elem_id='hd')
    photo = gr.Image(type='filepath', label='1️⃣ फ़ोटो डालो (कार्टून / कॉमिक / सीन)')
    prompt = gr.Textbox(lines=3, label='2️⃣ क्या हिलना चाहिए? (अंग्रेज़ी में लिखो तो बेहतर)',
                        placeholder='Example: the man slowly turns his head, rain falling, lightning flashes in the sky, camera slowly pushes in')
    quality = gr.Radio([('480p – तेज़ (4-6 मिनट)', '480'), ('720p – साफ़ (9-12 मिनट)', '720')], value='480', label='3️⃣ क्वालिटी')
    agree = gr.Checkbox(label='मैं मानता हूँ: असली लोगों का नक़ली वीडियो, बदनामी, अश्लील या हिंसक कंटेंट नहीं बनाऊँगा।')
    btn = gr.Button('🎬 वीडियो बनाओ', elem_id='go')
    msg = gr.Markdown('')
    vid = gr.Video(label='आपका वीडियो (5 सेकंड, 1080p)')
    btn.click(lambda: '⏳ बन रहा है... पहली बार मॉडल डाउनलोड होगा, थोड़ा ज़्यादा समय लगेगा। ये पेज बंद मत करना।', None, msg) \
       .then(banao, [photo, prompt, quality, agree], [vid, msg])
    gr.Markdown('---\n*हर वीडियो पर "AI Generated" लिखा आता है। Claude AI की मदद से बनाया गया। मॉडल: Wan 2.2 (Apache 2.0)।*')
print('\n🚀 नीचे https://....gradio.live वाला लिंक खोलो। ये सेल चलता रहने दो।\n')
app.queue().launch(share=True, debug=True)
